In [ ]:
EXPERIMENT_NAME = "brats2020_frozen_experiment"
SEED = 42

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

PATCH_SIZE = (128, 128, 128)

BATCH_SIZE = 1

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5

Imports & AMP

In [ ]:
import os
import json
import random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

Reproducibility Lock

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

Dataset Paths & Patient List

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
EXCLUDED_PATIENTS = {"BraTS20_Training_355"}

PATIENTS = sorted([
    pid for pid in os.listdir(DATASET_PATH)
    if pid.startswith("BraTS20_Training_")
    and pid not in EXCLUDED_PATIENTS
])

print("Total patients:", len(PATIENTS))

NIfTI Loading

In [ ]:
import nibabel as nib

def load_patient(pid):
    pdir = os.path.join(DATASET_PATH, pid)

    def load(fname):
        return nib.load(os.path.join(pdir, fname)).get_fdata()

    flair = load(f"{pid}_flair.nii")
    t1    = load(f"{pid}_t1.nii")
    t1ce  = load(f"{pid}_t1ce.nii")
    t2    = load(f"{pid}_t2.nii")
    seg   = load(f"{pid}_seg.nii")

    image = np.stack([t1, t1ce, t2, flair], axis=0)
    return image, seg

Normalization & Label Mapping

In [ ]:
def normalize(image):
    out = np.zeros_like(image)
    for c in range(image.shape[0]):
        x = image[c]
        mask = x > 0
        out[c] = (x - x[mask].mean()) / (x[mask].std() + 1e-8)
    return out

In [ ]:
def remap_labels(seg):
    out = np.zeros_like(seg, dtype=np.int64)
    out[seg == 1] = 1
    out[seg == 2] = 2
    out[seg == 4] = 3
    return out

Tumor-Aware Patch Sampler 128³

In [ ]:
def sample_patch(image, seg, tumor_prob=0.5):
    _, D, H, W = image.shape
    pd, ph, pw = PATCH_SIZE

    if np.random.rand() < tumor_prob and np.any(seg > 0):
        z, y, x = [v[np.random.randint(len(v))] for v in np.where(seg > 0)]
    else:
        z, y, x = np.random.randint(D), np.random.randint(H), np.random.randint(W)

    z1 = np.clip(z - pd//2, 0, D-pd)
    y1 = np.clip(y - ph//2, 0, H-ph)
    x1 = np.clip(x - pw//2, 0, W-pw)

    return (
        image[:, z1:z1+pd, y1:y1+ph, x1:x1+pw],
        seg[z1:z1+pd, y1:y1+ph, x1:x1+pw]
    )

Dataset Class

In [ ]:
class BraTS2020Dataset(Dataset):
    def __init__(self, patient_ids, patches_per_patient):
        self.patient_ids = patient_ids
        self.ppp = patches_per_patient

    def __len__(self):
        return len(self.patient_ids) * self.ppp

    def __getitem__(self, idx):
        pid = self.patient_ids[idx // self.ppp]
        image, seg = load_patient(pid)

        image = normalize(image)
        seg = remap_labels(seg)

        img, lab = sample_patch(image, seg)

        if np.random.rand() < 0.5:
            img = np.flip(img, axis=1).copy()
            lab = np.flip(lab, axis=0).copy()

        if np.random.rand() < 0.5:
            img = np.flip(img, axis=2).copy()
            lab = np.flip(lab, axis=1).copy()

        if np.random.rand() < 0.5:
            img = np.flip(img, axis=3).copy()
            lab = np.flip(lab, axis=2).copy()

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(lab, dtype=torch.long)
        )

Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    PATIENTS, test_size=0.2, random_state=SEED
)

print("Train:", len(train_ids), "Val:", len(val_ids))

DataLoaders

In [ ]:
train_loader = DataLoader(
    BraTS2020Dataset(train_ids, patches_per_patient=4),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    BraTS2020Dataset(val_ids, patches_per_patient=2),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

MODEL DEFINITION

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# 2. SEBlock3D
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y

In [ ]:
class AttentionGate3D(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.Wg = nn.Conv3d(F_g, F_int, 1, bias=False)
        self.Wx = nn.Conv3d(F_l, F_int, 1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv3d(F_int, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, g, x):
        return x * self.psi(self.Wg(g) + self.Wx(x))

In [ ]:
class UpAtt(nn.Module):
    def __init__(self, in_up, in_skip, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_up, out_ch, 2, stride=2)
        self.att = AttentionGate3D(out_ch, in_skip, out_ch//2)
        self.conv = DoubleConv(in_skip + out_ch, out_ch)

    def forward(self, x_up, x_skip):
        x_up = self.up(x_up)

        dz = x_skip.size(2) - x_up.size(2)
        dy = x_skip.size(3) - x_up.size(3)
        dx = x_skip.size(4) - x_up.size(4)

        x_up = F.pad(
            x_up,
            [dx//2, dx-dx//2, dy//2, dy-dy//2, dz//2, dz-dz//2]
        )

        x_skip = self.att(x_up, x_skip)
        return self.conv(torch.cat([x_skip, x_up], dim=1))

In [ ]:
class EfficientNet3DEncoder(nn.Module):
    def __init__(self, in_channels=4, base_c=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, base_c, 3, padding=1),
            nn.BatchNorm3d(base_c),
            nn.ReLU(inplace=True)
        )

        self.b1 = MBConv3D(base_c, base_c)
        self.b2 = MBConv3D(base_c, base_c*2, stride=2)
        self.b3 = MBConv3D(base_c*2, base_c*4, stride=2)
        self.b4 = MBConv3D(base_c*4, base_c*8, stride=2)
        self.b5 = MBConv3D(base_c*8, base_c*16, stride=2)

    def forward(self, x):
        x1 = self.stem(x)
        x2 = self.b1(x1)
        x3 = self.b2(x2)
        x4 = self.b3(x3)
        x5 = self.b4(x4)
        x6 = self.b5(x5)
        return x2, x3, x4, x5, x6

In [ ]:
class MBConv3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand=4):
        super().__init__()
        mid = in_ch * expand
        self.use_res = in_ch == out_ch and stride == 1

        self.block = nn.Sequential(
            nn.Conv3d(in_ch, mid, 1, bias=False),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),

            nn.Conv3d(mid, mid, 3, stride=stride, padding=1,
                      groups=mid, bias=False),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),

            SEBlock3D(mid),

            nn.Conv3d(mid, out_ch, 1, bias=False),
            nn.BatchNorm3d(out_ch)
        )

    def forward(self, x):
        return x + self.block(x) if self.use_res else self.block(x)

In [ ]:
class UNet_EfficientNet_Attention3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = EfficientNet3DEncoder()
        self.bot = DoubleConv(512, 1024)

        self.up4 = UpAtt(1024, 256, 512)
        self.up3 = UpAtt(512,  128, 256)
        self.up2 = UpAtt(256,   64, 128)
        self.up1 = UpAtt(128,   32, 64)

        self.outc = nn.Conv3d(64, 4, 1)

    def forward(self, x):
        x2, x3, x4, x5, x6 = self.encoder(x)
        x = self.bot(x6)
        x = self.up4(x, x5)
        x = self.up3(x, x4)
        x = self.up2(x, x3)
        x = self.up1(x, x2)
        return self.outc(x)

In [ ]:
model = UNet_EfficientNet_Attention3D().cuda()
model = nn.DataParallel(model)

In [ ]:
x = torch.randn(1, 4, 128, 128, 128).cuda()
with torch.no_grad():
    y = model(x)

print(y.shape)

Loss Functions

In [ ]:
ce_loss = nn.CrossEntropyLoss()

class DiceLoss(nn.Module):

    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.softmax(logits, dim=1)

        targets_onehot = F.one_hot(
            targets,
            num_classes=4
        ).permute(0,4,1,2,3).float()

        dims = (0,2,3,4)

        intersection = torch.sum(
            probs * targets_onehot,
            dims
        )

        cardinality = torch.sum(
            probs + targets_onehot,
            dims
        )

        dice = (
            2 * intersection + self.smooth
        ) / (
            cardinality + self.smooth
        )

        return 1 - dice.mean()

dice_criterion = DiceLoss()
ce_criterion = nn.CrossEntropyLoss()

def combined_loss(logits, targets):

    dice = dice_criterion(
        logits,
        targets
    )

    ce = ce_criterion(
        logits,
        targets
    )

    return dice + ce

Optimizer & AMP

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler()

Training & Validation Loops

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running = 0

    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()

        with autocast("cuda"):
            out = model(x)
            loss = combined_loss(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item()

    return running / len(loader)

In [ ]:
def segmentation_metrics(pred, target):

    pred = pred.flatten()
    target = target.flatten()

    tp = np.sum(
        (pred > 0) &
        (target > 0)
    )

    fp = np.sum(
        (pred > 0) &
        (target == 0)
    )

    fn = np.sum(
        (pred == 0) &
        (target > 0)
    )

    dice = (
        2 * tp
    ) / (
        2 * tp + fp + fn + 1e-8
    )

    iou = tp / (
        tp + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall
    }

In [ ]:
#Original
def validate(model, loader):

    model.eval()

    metrics = []

    with torch.no_grad():

        for x, y in loader:

            x = x.cuda()
            y = y.cuda()

            logits = model(x)

            pred = torch.argmax(
                logits,
                dim=1
            )

            m = segmentation_metrics(
                pred.cpu().numpy(),
                y.cpu().numpy()
            )

            metrics.append(m)

    return {
        key: np.mean(
            [m[key] for m in metrics]
        )
        for key in metrics[0]
    }

Training Loop + Early Stopping

In [ ]:
#original
best_dice = 0.0
patience = 0

epoch_history = []
train_loss_history = []
val_dice_history = []
val_iou_history = []

best_epoch = 0

for epoch in range(MAX_EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    metrics = validate(
        model,
        val_loader
    )

    val_dice = metrics["dice"]
    val_iou = metrics["iou"]
    val_precision = metrics["precision"]
    val_recall = metrics["recall"]

    epoch_history.append(epoch + 1)
    train_loss_history.append(train_loss)
    val_dice_history.append(val_dice)
    val_iou_history.append(val_iou)

    print(
        f"Epoch [{epoch+1}/{MAX_EPOCHS}] "
        f"Loss: {train_loss:.4f} | "
        f"Dice: {val_dice:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"Precision: {val_precision:.4f} | "
        f"Recall: {val_recall:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice
        best_epoch = epoch + 1
        patience = 0

        torch.save(
            model.module.state_dict(),
            "best_model.pth"
        )

    else:
        patience += 1

    if patience >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
SAVE_DIR = "/kaggle/working"
MODEL_NAME = "UNet_EfficientNet_Attention3D_brats2020"   

checkpoint = {
    "model_state_dict": model.module.state_dict(),
    "best_val_dice": best_dice,
    "best_epoch": best_epoch,
    "architecture": "3D U-Net + EfficientNet Encoder",
    "num_classes": 4,
    "input_channels": 4,
    "patch_size": [128, 128, 128],
    "seed": 42
}

torch.save(
    checkpoint,
    os.path.join(SAVE_DIR, f"{MODEL_NAME}.pth")
)

print(f"Model saved as {MODEL_NAME}.pth")

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "epoch": epoch_history,
    "train_loss": train_loss_history,
    "val_dice": val_dice_history,
    "val_iou": val_iou_history
})

results.to_csv(
    "/kaggle/working/training_metrics.csv",
    index=False
)

In [ ]:
training_logs = {
    "epochs": epoch_history,
    "train_loss": train_loss_history,
    "val_dice": val_dice_history,
    "val_iou": val_iou_history
}

log_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_logs.json")
with open(log_path, "w") as f:
    json.dump(training_logs, f, indent=4)

print(f"Training logs saved as {MODEL_NAME}_training_logs.json")

In [ ]:
{
    "experiment_name": "BraTS2020_Attention_EfficientNet_3D_UNet",
    "dataset": {
        "name": "BraTS 2020",
        "modalities": ["T1", "T1ce", "T2", "FLAIR"],
        "input_channels": 4,
        "excluded_patients": ["BraTS20_Training_355"],
        "label_mapping": {
            "0": "background",
            "1": "necrotic_core",
            "2": "edema",
            "3": "enhancing_tumor"
        }
    },
    "preprocessing": {
        "normalization": "per-modality z-score (non-zero voxels)",
        "patch_size": [128, 128, 128],
        "sampling_strategy": "tumor-aware random patch sampling",
        "tumor_sampling_probability": 0.5
    },
    "data_split": {
        "train_ratio": 0.8,
        "validation_ratio": 0.2,
        "split_strategy": "patient-wise",
        "random_seed": 42
    },
    "model": {
        "architecture": "Attention 3D U-Net with EfficientNet-style encoder",
        "encoder": {
            "type": "EfficientNet-style 3D encoder",
            "blocks": "MBConv3D with Squeeze-and-Excitation",
            "base_channels": 32,
            "depth": 5
        },
        "decoder": {
            "type": "3D U-Net decoder",
            "skip_connections": "attention-gated",
            "upsampling": "transposed convolution"
        },
        "bottleneck_channels": 1024,
        "output_classes": 4
    },
    "training": {
        "optimizer": "Adam",
        "learning_rate": 0.0001,
        "weight_decay": 0.00001,
        "batch_size_per_gpu": 1,
        "mixed_precision": True,
        "loss_function": "Dice + CrossEntropy",
        "max_epochs": 20,
        "early_stopping": {
            "enabled": True,
            "patience": 3,
            "monitor": "validation_dice"
        },
        "best_epoch": 4,
        "best_validation_dice": 0.6771
    },
    "hardware": {
        "platform": "Kaggle",
        "gpus": "2 × NVIDIA T4",
        "framework": "PyTorch"
    },
    "reproducibility": {
        "random_seed": 42,
        "deterministic_cudnn": True
    }
}